In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import umap
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
from openai import OpenAI

In [2]:
low_level_aspects = pd.read_csv('datasets/low_level_aspects.csv')['Aspects']

model = SentenceTransformer("sentence-transformers/all-distilroberta-v1")
aspects_embeddings = model.encode(low_level_aspects)

reducer = umap.UMAP(n_components=2, metric='cosine', random_state=42)
reduced = reducer.fit_transform(aspects_embeddings)

In [5]:
# Silhouette score — plot and save as PDF for paper
examine_clusters = 20

silhouette_scores = []
for num_clusters in range(2, examine_clusters+1):
    kmeans = KMeans(n_clusters=num_clusters, random_state=44).fit(reduced)
    labels = kmeans.labels_
    score = silhouette_score(reduced, labels)
    silhouette_scores.append(score)

import matplotlib as mpl
# adjust default font sizes for publication-quality figure
mpl.rcParams.update({'font.size': 12, 'axes.titlesize': 14, 'axes.labelsize': 14})

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(2, examine_clusters+1), silhouette_scores, marker='o', linewidth=1.5)
ax.set_xlabel('Number of clusters', fontsize=14)
ax.set_ylabel('Silhouette Score', fontsize=14)
ax.set_xticks(range(2, examine_clusters+1))
ax.grid(False)

# Save as vector PDF suitable for inclusion in papers
output_path = 'outputs/silhouette_scores.pdf'
plt.tight_layout()
fig.savefig(output_path, format='pdf', dpi=300, bbox_inches='tight')
print('Saved silhouette plot to', output_path)
fig

In [8]:
# K-means clustering
num_clusters = 13
kmeans = KMeans(n_clusters=num_clusters, random_state=42)
kmeans.fit(reduced)

# Get the labels and centroids
labels = kmeans.labels_
centroids = kmeans.cluster_centers_

import seaborn as sns
sns.set_style('whitegrid')

fig, ax = plt.subplots(figsize=(10, 7))

# Prepare dataframe with explicit UMAP dimension names
df = pd.DataFrame(reduced, columns=['UMAP-1', 'UMAP-2'])
df['label'] = labels

# Scatter colored by cluster label
sc = ax.scatter(df['UMAP-1'], df['UMAP-2'], c=df['label'], cmap='tab20', s=30, alpha=0.9)
ax.set_xlabel('UMAP-1', fontsize=14)
ax.set_ylabel('UMAP-2', fontsize=14)
# ax.set_title('K-means Clustering', fontsize=16)

# Remove numeric tick labels (numbers have no absolute meaning in UMAP space)
ax.set_xticks([])
ax.set_yticks([])

# Colorbar with integer ticks for cluster labels
cbar = plt.colorbar(sc, ax=ax, ticks=range(num_clusters))
cbar.set_label('label')

plt.tight_layout()

# Save as PDF for paper
output_path = 'outputs/kmeans_clusters.pdf'
fig.savefig(output_path, format='pdf', dpi=300, bbox_inches='tight')
print('Saved k-means cluster plot to', output_path)
fig

In [8]:
# Sumarize by using LLMs with prompt
client = OpenAI(
    api_key=""
    )

def pipeline_gpt(msg, client):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", 
            "content": "You are a professional annotator. Please summarize the following aspects related to campus mental health services into single aspect that is the most representative of the cluster, with at most 3 words."
            },

            {
                "role": "user",
                "content": msg
            }
        ],
        temperature=1.0,
        top_p=1.0
    )

    return response.choices[0].message.content

In [ ]:
clustered_dict = dict()
for i, label in enumerate(labels):
    label = str(label)
    if label not in clustered_dict:
        clustered_dict[label] = [low_level_aspects[i]]
    else:
        clustered_dict[label].append(low_level_aspects[i])

clustered_dict = {k: v for k, v in sorted(clustered_dict.items(), key=lambda item: int(item[0]))}

responses = []
for i in range(len(clustered_dict)):
    i = str(i)
    msg = ", ".join(clustered_dict[i])
    response = pipeline_gpt(msg, client)
    print("Cluster", i, ":", response)
    responses.append(response)
clustered_dict_with_responses = {responses[int(k)]: v for k, v in clustered_dict.items() if k != '-1'}